In [ ]:
# Setup: garante que .env seja carregado e pacote esteja no path
import sys, os
from pathlib import Path

# Localiza raiz do projeto (sobe até encontrar pyproject.toml)
project_root = Path(os.getcwd())
for p in [project_root, *project_root.parents]:
    if (p / 'pyproject.toml').exists():
        project_root = p
        break

# Adiciona src/ ao sys.path (caso o pacote não esteja instalado em editable mode)
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Carrega .env da raiz do projeto
from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

print(f'Project root: {project_root}')
print(f'FRED_API_KEY: {"SET" if os.getenv("FRED_API_KEY") else "NOT SET — verifique o .env"}')

# Analise de Renda Fixa — Tesouro Direto

Analise de titulos publicos, curva de juros reais e comparativo com benchmarks.

Fontes:
- **TesouroDiretoFetcher**: taxas atuais, historico NTN-B, curva de juros reais
- **BCBFetcher**: Selic, CDI, IPCA (contexto para renda fixa)

**Nota**: Nao requer API keys. Dados publicos do Tesouro Nacional e BCB.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Taxas Atuais — Tesouro Direto

In [ ]:
from carteira_auto.data.fetchers import TesouroDiretoFetcher
import pandas as pd

tesouro = TesouroDiretoFetcher()
taxas = pd.DataFrame()  # fallback vazio
curva = pd.DataFrame()  # fallback vazio

# Taxas atuais — Tesouro Transparente pode retornar 503 em picos de tráfego.
# Em caso de falha, as células seguintes usam os DataFrames vazios sem travar.
try:
    taxas = tesouro.get_current_rates()
    print(f"Titulos disponíveis: {len(taxas)}")
    print()
    print(taxas.to_string(index=False))
except Exception as e:
    print(f"[AVISO] TesouroDireto get_current_rates falhou: {e}")
    print("Servidor pode estar temporariamente indisponível. Tente novamente em alguns minutos.")
    print("URL: https://www.tesourotransparente.gov.br/ckan/...")


In [ ]:
# Visualizar taxas por tipo de título
# Colunas reais: tipo, vencimento (str), data, taxa_compra, taxa_venda, pu_compra, pu_venda
if not taxas.empty and "taxa_compra" in taxas.columns:
    fig, ax = plt.subplots(figsize=(12, 6))

    taxas_plot = taxas.dropna(subset=["taxa_compra"]).copy()

    # vencimento chega como string ('01/01/2029') — converter para datetime para extrair o ano
    if "vencimento" in taxas_plot.columns:
        venc_dt = pd.to_datetime(taxas_plot["vencimento"], dayfirst=True, errors="coerce")
        year_str = venc_dt.dt.year.astype("Int64").astype(str).str.replace("<NA>", "")
        taxas_plot["label"] = taxas_plot["tipo"].astype(str) + " " + year_str
    else:
        taxas_plot["label"] = taxas_plot["tipo"].astype(str)

    taxas_sorted = taxas_plot.sort_values("taxa_compra", ascending=True)

    cores_titulo = []
    for tipo in taxas_sorted["tipo"]:
        if "NTN-B" in str(tipo) or "IPCA" in str(tipo):
            cores_titulo.append("#2ca02c")
        elif "LFT" in str(tipo) or "Selic" in str(tipo):
            cores_titulo.append("#1f77b4")
        else:
            cores_titulo.append("#ff7f0e")

    ax.barh(range(len(taxas_sorted)), taxas_sorted["taxa_compra"],
            color=cores_titulo, alpha=0.85, height=0.6)
    ax.set_yticks(range(len(taxas_sorted)))
    ax.set_yticklabels(taxas_sorted["label"], fontsize=9)
    ax.set_title("Taxas de Compra — Tesouro Direto", fontsize=13, fontweight="bold")
    ax.set_xlabel("Taxa (% a.a.)")

    for i, (_, row) in enumerate(taxas_sorted.iterrows()):
        ax.text(row["taxa_compra"] + 0.05, i, f"{row['taxa_compra']:.2f}%",
                va="center", fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print(f"Colunas disponíveis: {list(taxas.columns) if not taxas.empty else 'DataFrame vazio'}")


## 2. Curva NTN-B — Juros Reais

A curva NTN-B mostra as taxas reais (acima do IPCA) para diferentes vencimentos.
Util para avaliar o premio de risco e a expectativa de inflacao.

In [ ]:
# Curva NTN-B (juros reais por vencimento)
# get_ntnb_curve() retorna DataFrame com colunas: vencimento, taxa_compra, taxa_venda, ...
try:
    curva = tesouro.get_ntnb_curve()
    print("Curva NTN-B (juros reais):")
    print(curva[["vencimento", "taxa_compra", "taxa_venda"]].to_string(index=False) if not curva.empty else "Sem dados")
except Exception as e:
    curva = pd.DataFrame()
    print(f"[AVISO] get_ntnb_curve falhou: {e}")

if not curva.empty and "taxa_compra" in curva.columns and "vencimento" in curva.columns:
    curva_plot = curva.dropna(subset=["taxa_compra", "vencimento"]).copy()
    labels_venc = [str(v.year) if hasattr(v, 'year') else str(v) for v in curva_plot["vencimento"]]
    fig, ax = plt.subplots()
    ax.plot(labels_venc, curva_plot["taxa_compra"].values, "o-",
            markersize=8, linewidth=2, color="#d62728", label="Taxa Compra")
    ax.plot(labels_venc, curva_plot["taxa_venda"].values, "s--",
            markersize=6, linewidth=1.5, color="#1f77b4", label="Taxa Venda", alpha=0.7)
    ax.set_title("Curva de Juros Reais (NTN-B)")
    ax.set_xlabel("Vencimento")
    ax.set_ylabel("Taxa real (% a.a.)")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Gráfico NTN-B indisponível — curva vazia ou API fora do ar.")


## 3. Historico NTN-B Principal

In [ ]:
# Historico de precos da NTN-B Principal
ntnb = pd.DataFrame()
try:
    ntnb = tesouro.get_ntnb_history(com_cupom=False)
    print(f"NTN-B Principal historico: {len(ntnb)} registros")
    if not ntnb.empty:
        print(ntnb.tail(5))
except Exception as e:
    print(f"[AVISO] get_ntnb_history falhou: {e}")


## 4. Contexto — Selic, CDI e IPCA

Para avaliar renda fixa, comparamos com os benchmarks de referencia.

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()

selic = bcb.get_selic(period_days=30)
ipca = bcb.get_ipca(period_days=365)

selic_atual = selic["valor"].iloc[-1]
ipca_12m = ((1 + ipca.tail(12)["valor"] / 100).prod() - 1) * 100

print(f"Selic meta: {selic_atual:.2f}% a.a.")
print(f"IPCA acumulado 12m: {ipca_12m:.2f}%")
print(f"Juro real (Selic - IPCA): {selic_atual - ipca_12m:.2f}% a.a.")

In [ ]:
# Comparativo visual de rentabilidades
fig, ax = plt.subplots(figsize=(8, 5))

labels_rf = ["Selic nominal", "Selic real", "Poupança\n(≤70% Selic)"]
valores_rf = [selic_atual, selic_atual - ipca_12m, selic_atual * 0.7]

if not curva.empty and "taxa_compra" in curva.columns:
    labels_rf.append("NTN-B longa\n(real)")
    valores_rf.append(float(curva["taxa_compra"].iloc[-1]))  # taxa escalar

cores_rf = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"][:len(labels_rf)]

bars = ax.bar(labels_rf, valores_rf, color=cores_rf, alpha=0.85, width=0.5)
for bar, val in zip(bars, valores_rf, strict=False):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"{val:.2f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Comparativo de Rentabilidade (% a.a.)", fontsize=13, fontweight="bold")
ax.set_ylabel("% a.a.")
ax.axhline(y=0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# Comparativo: NTN-B vs Selic real
print("\n=== Comparativo de Rentabilidade ===")
print(f"Selic nominal: {selic_atual:.2f}% a.a.")
print(f"Selic real (- IPCA): {selic_atual - ipca_12m:.2f}% a.a.")
if not curva.empty and "taxa_compra" in curva.columns:
    ntnb_longa = float(curva["taxa_compra"].iloc[-1])  # escalar, nao Series
    print(f"NTN-B longa (juro real): {ntnb_longa:.2f}% a.a.")
    print(f"\nNTN-B longa vs Selic real: {ntnb_longa - (selic_atual - ipca_12m):+.2f} pp")


## Resumo

| Titulo | Tipo | Referencia | Melhor para |
|--------|------|------------|-------------|
| Tesouro Selic | Pos-fixado | Selic | Liquidez, reserva de emergencia |
| Tesouro IPCA+ | Inflacao + real | IPCA + taxa | Protecao inflacionaria, longo prazo |
| Tesouro Prefixado | Taxa fixa | CDI | Cenario de queda de juros |